# Week 07 Puzzles — NumPy, Pandas & Matplotlib

> **NS5116 Computational Neuroscience — Spring 2026**

This notebook covers the three core data-science libraries you will use throughout the rest of the course:

| Library | Role |
| ------- | ---- |
| **NumPy** | Fast numerical arrays, vectorized math, statistics |
| **Pandas** | Tabular data with labels (DataFrames), CSV I/O, groupby |
| **Matplotlib** | Static scientific figures — line, scatter, bar, histogram |

- **Part 1 — Guided Practice (Puzzles 1–10):** Worked solutions. Run, study, and modify.
- **Part 2 — Independent Practice (Puzzles 11–20):** Empty code cells — write your own solution.

---

## Part 1 — Guided Practice (with solutions)

### Puzzle 1 — NumPy: Create arrays and compute descriptive statistics

Simulate 30 reaction times from a normal distribution (mean = 420 ms, std = 75 ms) with seed 42.

Print:
- The first 5 values
- Mean, standard deviation, min, and max
- The 25th and 75th percentiles (interquartile range)

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
rts = rng.normal(loc=420, scale=75, size=30)

print(f"First 5 RTs: {rts[:5].round(1)}")
print(f"Mean:  {rts.mean():.1f} ms")
print(f"Std:   {rts.std():.1f} ms")
print(f"Min:   {rts.min():.1f} ms")
print(f"Max:   {rts.max():.1f} ms")

q25, q75 = np.percentile(rts, [25, 75])
print(f"IQR:   {q25:.1f} – {q75:.1f} ms")

### Puzzle 2 — NumPy: Boolean masking and z-score normalization

Given the 30 RTs from Puzzle 1:

1. Remove outliers: keep only RTs within **±2 SD** of the mean.
2. Z-score normalize the cleaned array.
3. Print how many trials were removed and verify the z-score mean ≈ 0 and std ≈ 1.

In [ ]:
mean, std = rts.mean(), rts.std()

# Boolean mask: keep RTs within ±2 SD
mask = np.abs(rts - mean) <= 2 * std
rts_clean = rts[mask]

print(f"Original: {len(rts)} trials | Cleaned: {len(rts_clean)} | Removed: {len(rts) - len(rts_clean)}")

# Z-score
rts_z = (rts_clean - rts_clean.mean()) / rts_clean.std()
print(f"Z-score mean: {rts_z.mean():.6f}  (should be ~0)")
print(f"Z-score std:  {rts_z.std():.6f}  (should be ~1)")

### Puzzle 3 — NumPy: 2D array and axis-wise statistics

Simulate a **3 subjects × 20 trials** matrix of RTs.

Compute:
1. Mean RT per subject (collapse over trials, `axis=1`)
2. Mean RT per trial position (collapse over subjects, `axis=0`)
3. The subject with the fastest average RT

In [ ]:
rng = np.random.default_rng(7)
data = rng.normal(loc=[400, 430, 410], scale=60, size=(20, 3)).T  # shape: (3, 20)

per_subject = data.mean(axis=1)   # mean across 20 trials for each subject
per_trial   = data.mean(axis=0)   # mean across 3 subjects at each trial position

print("Mean RT per subject:", per_subject.round(1))
print("Mean RT per trial (first 5):", per_trial[:5].round(1))

fastest = np.argmin(per_subject) + 1   # 1-indexed subject number
print(f"Fastest subject: S{fastest} ({per_subject.min():.1f} ms)")

### Puzzle 4 — NumPy: `np.where` and vectorized labelling

Given accuracy scores for 10 participants, label each as `"high"` (≥ 0.85) or `"low"` (< 0.85) using `np.where`.

Then compute:
- Mean accuracy for each group
- Count of participants in each group

In [ ]:
accuracy = np.array([0.92, 0.78, 0.85, 0.91, 0.73, 0.88, 0.80, 0.95, 0.82, 0.87])

labels = np.where(accuracy >= 0.85, "high", "low")
print("Labels:", labels)

high_mask = labels == "high"
low_mask  = labels == "low"

print(f"High accuracy: n={high_mask.sum()}, mean={accuracy[high_mask].mean():.3f}")
print(f"Low  accuracy: n={low_mask.sum()},  mean={accuracy[low_mask].mean():.3f}")

### Puzzle 5 — Pandas: Create a DataFrame from experiment data

Build a DataFrame representing 12 trials from a Stroop task with columns:
`subject`, `trial`, `condition`, `rt_ms`, `correct`.

Print the DataFrame and its basic info (`dtypes`, shape).

In [ ]:
import pandas as pd

data = {
    "subject":   ["P01"] * 6 + ["P02"] * 6,
    "trial":     list(range(1, 7)) * 2,
    "condition": ["congruent", "incongruent"] * 6,
    "rt_ms":     [320, 450, 310, 480, 330, 460,
                  340, 470, 300, 490, 350, 455],
    "correct":   [1, 1, 1, 0, 1, 1,
                  1, 1, 1, 1, 0, 1],
}

df = pd.DataFrame(data)

print(df)
print(f"\nShape: {df.shape}")
print("\nDtypes:")
print(df.dtypes)

### Puzzle 6 — Pandas: Filter rows and select columns

Using the DataFrame from Puzzle 5:

1. Keep only correct trials (`correct == 1`).
2. From those, keep only trials where `rt_ms < 400`.
3. Print the resulting `subject`, `condition`, and `rt_ms` columns.

In [ ]:
# Filter correct trials with RT < 400 ms
df_fast_correct = df[(df["correct"] == 1) & (df["rt_ms"] < 400)]

print(df_fast_correct[["subject", "condition", "rt_ms"]])

### Puzzle 7 — Pandas: groupby and aggregation

Using the DataFrame from Puzzle 5, compute for **each subject × condition** combination:
- Mean RT (correct trials only)
- Accuracy (proportion correct)

Print both summary tables.

In [ ]:
# Mean RT on correct trials only
mean_rt = (
    df[df["correct"] == 1]
    .groupby(["subject", "condition"])["rt_ms"]
    .mean()
    .round(1)
)

# Accuracy = proportion correct
accuracy = (
    df
    .groupby(["subject", "condition"])["correct"]
    .mean()
    .round(3)
)

print("Mean RT (correct trials):\n", mean_rt)
print("\nAccuracy:\n", accuracy)

### Puzzle 8 — Pandas: Read and write CSV

1. Save the DataFrame from Puzzle 5 to `stroop_data.csv` (no index column).
2. Read it back into a new DataFrame.
3. Confirm the shape and first 3 rows match the original.

In [ ]:
# Save to CSV
df.to_csv("stroop_data.csv", index=False)
print("Saved stroop_data.csv")

# Read back
df_loaded = pd.read_csv("stroop_data.csv")

print(f"\nLoaded shape: {df_loaded.shape}  (original: {df.shape})")
print("\nFirst 3 rows:")
print(df_loaded.head(3))

### Puzzle 9 — Matplotlib: Line plot and histogram from NumPy data

Simulate 50 RTs (mean = 400, std = 70, seed = 42). Create a **1 × 2** figure:

- **Left panel:** Line plot of RT across trials with a red dashed mean reference line.
- **Right panel:** Histogram (20 bins, density=True) with a vertical mean line.

Add titles, axis labels, and `plt.tight_layout()`.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
trials = np.arange(1, 51)
rts_sim = rng.normal(loc=400, scale=70, size=50)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: line plot
axes[0].plot(trials, rts_sim, color="steelblue", linewidth=1.5, label="RT")
axes[0].axhline(rts_sim.mean(), color="red", linestyle="--", label=f"Mean = {rts_sim.mean():.0f} ms")
axes[0].set_xlabel("Trial number")
axes[0].set_ylabel("RT (ms)")
axes[0].set_title("RT across trials")
axes[0].legend()

# Right: histogram
axes[1].hist(rts_sim, bins=20, density=True, color="steelblue", edgecolor="white", alpha=0.85)
axes[1].axvline(rts_sim.mean(), color="red", linestyle="--", label=f"Mean = {rts_sim.mean():.0f} ms")
axes[1].set_xlabel("RT (ms)")
axes[1].set_ylabel("Probability density")
axes[1].set_title("RT distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

### Puzzle 10 — Matplotlib: Bar chart from Pandas groupby result

Using the Stroop DataFrame from Puzzle 5, compute mean RT per condition across all subjects and trials (correct trials only). Plot a bar chart with error bars (standard error of the mean).

- Use distinct colors for each condition.
- Include `capsize=5` on error bars.
- Y-axis starts at 0.

In [ ]:
df_correct = df[df["correct"] == 1]

grouped = df_correct.groupby("condition")["rt_ms"]
means = grouped.mean()
sems  = grouped.sem()   # standard error of the mean

conditions = means.index.tolist()
colors = ["#4C72B0", "#DD8452"]

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(conditions, means, yerr=sems, capsize=5,
       color=colors, edgecolor="black", linewidth=1.0,
       width=0.5, alpha=0.85)
ax.set_ylabel("Mean RT (ms)")
ax.set_title("Stroop Effect — Mean RT by Condition")
ax.set_ylim(0, max(means) * 1.4)
plt.tight_layout()
plt.show()

---

## Part 2 — Independent Practice (write your own solutions)

The cells below contain only problem descriptions. Write your solution in the provided code cell.
There is no single correct answer — focus on clarity, correctness, and proper scientific practice.

### Puzzle 11 — NumPy: Broadcasting across conditions

You have a **(2 conditions × 5 trials)** RT matrix and per-condition baselines:

```python
rts = np.array([[320, 415, 280, 510, 390],   # congruent
                [350, 450, 310, 540, 420]])   # incongruent
baselines = np.array([400, 450])             # one per condition
```

Without a loop, subtract each condition's baseline from its row (broadcasting). Print the centered matrix.

In [ ]:
# Your solution here

### Puzzle 12 — NumPy: Outlier removal with 2.5 SD threshold

Given:
```python
rts = np.array([320, 415, 280, 510, 390, 45, 1950, 450, 370, 400])
```

Remove trials whose RT is more than **2.5 standard deviations** from the mean. Print:
- Original count and cleaned count
- Original and cleaned means
- The indices of removed trials

In [ ]:
# Your solution here

### Puzzle 13 — NumPy: Correlation between accuracy and mean RT

You have 10 subjects with accuracy and mean RT data:

```python
accuracy = np.array([0.90, 0.85, 0.88, 0.92, 0.80, 0.87, 0.91, 0.83, 0.89, 0.86])
mean_rts = np.array([340, 380, 360, 330, 420, 370, 345, 395, 355, 375])
```

Use `np.corrcoef` to compute the Pearson correlation between accuracy and mean RT. Extract and print the correlation coefficient (not the full matrix).

In [ ]:
# Your solution here

### Puzzle 14 — NumPy: argsort to rank subjects by performance

Given mean RTs for 8 subjects:

```python
subject_ids = ["S01", "S02", "S03", "S04", "S05", "S06", "S07", "S08"]
mean_rts    = np.array([390, 415, 360, 470, 340, 425, 380, 445])
```

Use `np.argsort` to rank subjects from **fastest to slowest**. Print a ranked table showing rank, subject ID, and mean RT.

In [ ]:
# Your solution here

### Puzzle 15 — Pandas: Add a derived column and filter

Using the `df` DataFrame from Puzzle 5:

1. Add a new column `"rt_sec"` that converts `rt_ms` to seconds.
2. Add a column `"speed"` that labels each trial as `"fast"` (rt_ms < 380) or `"slow"` (rt_ms ≥ 380).
3. Print all slow trials for subject P01.

In [ ]:
# Your solution here

### Puzzle 16 — Pandas: pivot_table

Using the `df` DataFrame from Puzzle 5, create a **pivot table** showing the mean RT for each subject (rows) × condition (columns) combination (correct trials only).

Hint: use `pd.pivot_table(df_correct, values="rt_ms", index="subject", columns="condition", aggfunc="mean")`

In [ ]:
# Your solution here

### Puzzle 17 — Pandas: apply a custom function per group

Using the `df` DataFrame from Puzzle 5, compute the **interference cost** for each subject:

```
interference cost = mean_RT(incongruent) − mean_RT(congruent)
```

Use `groupby` + `apply` (or `pivot_table` → subtraction). Print the cost for each subject.

In [ ]:
# Your solution here

### Puzzle 18 — Pandas: merge two DataFrames

You have a second DataFrame with subject metadata:

```python
meta = pd.DataFrame({
    "subject": ["P01", "P02"],
    "age":     [22, 25],
    "group":   ["control", "patient"],
})
```

Merge `df` and `meta` on `"subject"`. Print the merged DataFrame and confirm no rows were lost.

In [ ]:
# Your solution here

### Puzzle 19 — Matplotlib: Scatter plot with annotation

For 10 subjects, plot mean RT (x-axis) vs. accuracy (y-axis) as a scatter plot.

Use the data from Puzzle 13:
```python
accuracy = np.array([0.90, 0.85, 0.88, 0.92, 0.80, 0.87, 0.91, 0.83, 0.89, 0.86])
mean_rts = np.array([340, 380, 360, 330, 420, 370, 345, 395, 355, 375])
```

- Color the fastest subject (lowest RT) in **red** and all others in **steelblue**.
- Annotate the fastest subject with the text `"Fastest"`.
- Add axis labels and a title.

In [ ]:
# Your solution here

### Puzzle 20 — Matplotlib: 2×2 summary figure from Pandas data

Using `df` from Puzzle 5, create a **2 × 2** figure:

- **Top-left:** Line plot of RT across trials for subject P01 (congruent = blue, incongruent = orange).
- **Top-right:** Overlapping histograms of RT for the two conditions (all subjects, density=True, alpha=0.5).
- **Bottom-left:** Bar chart of mean RT by condition across all subjects (with SEM error bars).
- **Bottom-right:** Bar chart of accuracy by condition across all subjects.

Add a figure-level title `"Stroop Task Summary"`.

In [ ]:
# Your solution here